# Week 34, Day 3: SOLID: L, I and D

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w034d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        self.balance -= amount
        return self.balance

class FixedSaver(Account):
    def withdraw(self, amount):
        raise RuntimeError("no withdrawals until 2027")

def pay_rent(account):
    return account.withdraw(500)

for acc in [Account(1000), FixedSaver(1000)]:
    try:
        print(pay_rent(acc))
    except RuntimeError as err:
        print("RuntimeError:", err)

In [ ]:
class WithdrawalRefused(Exception):
    """Raised when an account won't allow a withdrawal."""

class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        """Take amount out and return the new balance. Raises WithdrawalRefused if not allowed."""
        if amount > self.balance:
            raise WithdrawalRefused(f"only {self.balance} available")
        self.balance -= amount
        return self.balance

class FixedSaver(Account):
    def withdraw(self, amount):
        raise WithdrawalRefused("no withdrawals until 2027")

def pay_rent(account):
    try:
        return f"paid, {account.withdraw(500)} left"
    except WithdrawalRefused as err:
        return f"rent not paid: {err}"

for acc in [Account(1000), Account(200), FixedSaver(1000)]:
    print(pay_rent(acc))

In [ ]:
class Rectangle:
    def __init__(self, w, h):
        self.w, self.h = w, h

    def set_width(self, w):
        self.w = w

    def area(self):
        return self.w * self.h

class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)

    def set_width(self, w):              # a square must stay square...
        self.w = self.h = w

def stretch(r):
    r.set_width(10)
    return r.area()                      # ...but callers expect only the width to change

print(stretch(Rectangle(2, 3)), stretch(Square(3)))

In [ ]:
from abc import ABC, abstractmethod

class OfficeDevice(ABC):
    @abstractmethod
    def print_doc(self, text): ...
    @abstractmethod
    def scan(self): ...
    @abstractmethod
    def fax(self, number, text): ...

class HomePrinter(OfficeDevice):
    def print_doc(self, text):
        return f"printed {len(text)} characters"
    def scan(self):
        raise NotImplementedError("this printer can't scan")
    def fax(self, number, text):
        raise NotImplementedError("this printer can't fax")

print(HomePrinter().print_doc("Hello"))

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Printer(Protocol):
    def print_doc(self, text: str) -> str: ...

@runtime_checkable
class Scanner(Protocol):
    def scan(self) -> str: ...

class HomePrinter:
    def print_doc(self, text):
        return f"printed {len(text)} characters"

class OfficeMachine:
    def print_doc(self, text):
        return f"office printed {len(text)} characters"
    def scan(self):
        return "scanned page"

def print_all(printer: Printer, docs: list[str]) -> list[str]:
    return [printer.print_doc(d) for d in docs]

print(print_all(HomePrinter(), ["Hi", "Bye"]), print_all(OfficeMachine(), ["Hi"]))
print(isinstance(HomePrinter(), Scanner), isinstance(OfficeMachine(), Scanner))

In [ ]:
from datetime import datetime

class Reminders:
    def __init__(self, clock, send):
        self.clock = clock               # anything callable that returns the current time
        self.send = send                 # anything callable that sends a text
        self.items = []

    def add(self, when, text):
        self.items.append((when, text))

    def run(self):
        now = self.clock()
        due = [(w, t) for w, t in self.items if w <= now]
        self.items = [(w, t) for w, t in self.items if w > now]
        for _, text in sorted(due):
            self.send(text)
        return len(due)

# In a test: a fake clock and a fake sender.
sent = []
fake_now = datetime(2026, 11, 3, 9, 1)
r = Reminders(clock=lambda: fake_now, send=sent.append)
r.add(datetime(2026, 11, 3, 9, 0), "take medicine")
r.add(datetime(2026, 11, 3, 18, 0), "call Kemi")
print(r.run(), sent, len(r.items))

> **Key idea.** Liskov substitution (Barbara Liskov): code that works with a class must work with every subclass, so children accept what the parent accepts, return what it promises, raise only expected errors and keep its rules; fix violations in the design, not with `isinstance` checks. Interface segregation: depend only on what you use, with small interfaces such as one-ability Protocols. Dependency inversion: high-level rules shouldn't name low-level details; pass in the clock, storage and senders as arguments, create the real ones once at the edge of the program, and use simple fakes in tests.

### Exercise: A refusal every caller expects

Fix the design so that `FixedSaver` keeps `Account`'s promise. Add an exception `WithdrawalRefused`; make `Account.withdraw` raise it when the amount is more than the balance; make `FixedSaver.withdraw` raise it too (with any message); and change `pay_rent` to return `"paid, 500 left"` or `"rent not paid: ..."` with the exception's message. `pay_rent` must not mention `FixedSaver`.

In [ ]:
class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        self.balance -= amount
        return self.balance

class FixedSaver(Account):
    def withdraw(self, amount):
        raise RuntimeError("no withdrawals until 2027")

def pay_rent(account):
    return account.withdraw(500)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_refused():\n    \"substitutable\"\n    import ast\n    A, F, W = ns[\"Account\"], ns[\"FixedSaver\"], ns[\"WithdrawalRefused\"]\n    assert issubclass(W, Exception) and pay_rent(A(1000)) == \"paid, 500 left\"\n    a = A(200)\n    assert pay_rent(a) == \"rent not paid: \" + str(_err(lambda: A(200).withdraw(500), W)) and a.balance == 200\n    assert pay_rent(F(5000)).startswith(\"rent not paid: \")\n    _err(lambda: F(10).withdraw(1), W)\n    body = next(ast.get_source_segment(source, n) for n in ast.walk(ast.parse(source)) if isinstance(n, ast.FunctionDef) and n.name == \"pay_rent\")\n    assert \"FixedSaver\" not in body, \"pay_rent shouldn't need to know about FixedSaver\"\ndef _err(f, kind):\n    try:\n        f()\n    except kind as e:\n        return e\n    assert False, f\"should raise {kind.__name__}\"")

### Exercise: Not really a list

`ReadOnlyList` inherits from `list` but refuses to change, so any code that appends to a list breaks when given one. Replace it with a class `Frozen(items)` that doesn't inherit from `list` and offers only reading: `len`, indexing (including slices, which give a plain `list`), `in`, iteration, and a `repr` like `Frozen([1, 2, 3])`.

In [ ]:
class ReadOnlyList(list):
    def append(self, item):
        raise TypeError("read-only")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_frozen():\n    \"only reading\"\n    F = ns[\"Frozen\"]\n    f = F([3, 1, 2])\n    assert len(f) == 3 and f[0] == 3 and f[-1] == 2 and f[0:2] == [3, 1] and 1 in f and list(f) == [3, 1, 2] and sorted(f) == [1, 2, 3]\n    assert repr(f) == \"Frozen([3, 1, 2])\" and not issubclass(F, list) and not hasattr(f, \"append\") and \"ReadOnlyList\" not in ns\n    src = [1]\n    g = F(src)\n    src.append(2)\n    assert len(g) == 1, \"keep your own copy\"")

### Exercise: Small interfaces

Replace the large `OfficeDevice` ABC with three runtime-checkable Protocols, `Printer` (`print_doc(text) -> str`), `Scanner` (`scan() -> str`) and `Fax` (`fax(number, text) -> str`). Rewrite `HomePrinter` so it only prints, and `OfficeMachine` so it does all three, neither inheriting from anything. Then write `print_all(printer, docs)` (a list of results) and `capabilities(device)`, giving the sorted names of the Protocols the device matches, such as `["Fax", "Printer", "Scanner"]`.

In [ ]:
from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

class OfficeDevice(ABC):
    @abstractmethod
    def print_doc(self, text): ...
    @abstractmethod
    def scan(self): ...
    @abstractmethod
    def fax(self, number, text): ...

class HomePrinter(OfficeDevice):
    def print_doc(self, text):
        return f"printed {len(text)}"
    def scan(self):
        raise NotImplementedError
    def fax(self, number, text):
        raise NotImplementedError

class OfficeMachine(OfficeDevice):
    def print_doc(self, text):
        return f"office printed {len(text)}"
    def scan(self):
        return "scanned"
    def fax(self, number, text):
        return f"faxed {number}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_abilities():\n    \"small interfaces\"\n    P, S, F, H, O = (ns[n] for n in [\"Printer\", \"Scanner\", \"Fax\", \"HomePrinter\", \"OfficeMachine\"])\n    assert \"OfficeDevice\" not in ns, \"remove the big interface\"\n    h, o = H(), O()\n    assert not hasattr(h, \"scan\") and not hasattr(h, \"fax\") and H.__bases__ == (object,) and O.__bases__ == (object,)\n    assert print_all(h, [\"ab\", \"c\"]) == [\"printed 2\", \"printed 1\"] and print_all(o, [\"abc\"]) == [\"office printed 3\"]\n    assert capabilities(h) == [\"Printer\"] and capabilities(o) == [\"Fax\", \"Printer\", \"Scanner\"] and capabilities(\"x\") == []\n    assert o.fax(\"0113\", \"hi\") == \"faxed 0113\" and isinstance(o, S) and not isinstance(h, F)")

### Exercise: Passing in the clock

`greet` reads the real clock, so it can't be tested for the morning, afternoon and evening on demand. Change it to `greet(name, now=None)`, where `now` is a function returning a `datetime`, defaulting to `datetime.now` when not given. Then write a class `Shop(clock)` with `is_open()`, open from 8:00 up to (not including) 20:00 Monday to Saturday and 10:00 to 16:00 on Sunday, using only the clock it was given.

In [ ]:
from datetime import datetime

def greet(name):
    hour = datetime.now().hour
    if hour < 12:
        return f"Good morning, {name}"
    if hour < 18:
        return f"Good afternoon, {name}"
    return f"Good evening, {name}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_greet():\n    \"a clock passed in\"\n    from datetime import datetime as dt\n    at = lambda *a: (lambda: dt(*a))\n    assert greet(\"Ada\", at(2026, 11, 3, 9, 0)) == \"Good morning, Ada\" and greet(\"Ada\", at(2026, 11, 3, 12, 0)) == \"Good afternoon, Ada\"\n    assert greet(\"Bo\", at(2026, 11, 3, 18, 0)) == \"Good evening, Bo\" and greet(\"Bo\").startswith(\"Good \")\n    S = ns[\"Shop\"]\n    tue = [S(at(2026, 11, 3, h, m)).is_open() for h, m in [(7, 59), (8, 0), (19, 59), (20, 0)]]\n    sun = [S(at(2026, 11, 8, h, m)).is_open() for h, m in [(9, 59), (10, 0), (15, 59), (16, 0)]]\n    assert tue == [False, True, True, False] and sun == [False, True, True, False], f\"{tue} {sun}\"\n    assert \"datetime.now()\" not in source, \"call the clock you're given\"")

### Bug hunt: A child that returns the wrong thing

`total_fees` was written for payments whose `fee()` returns pence, but the new `Overseas` payment returns text, so it crashes with `TypeError`. Fix `Overseas` so it keeps the promise (a fee of 250 pence) and the program prints `280`.

In [ ]:
class Payment:
    def __init__(self, pence):
        self.pence = pence

    def fee(self):
        """The fee in pence, as an int."""
        return 0

class Card(Payment):
    def fee(self):
        return self.pence * 15 // 1000

class Overseas(Payment):
    def fee(self):
        return "£2.50"

def total_fees(payments):
    return sum(p.fee() for p in payments)

print(total_fees([Card(2000), Overseas(100), Payment(1)]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the fees\"\n    expect_output(\"280\")")

### Bug hunt: The clock that's ignored

`Library` takes a clock so its tests can choose the date, but `is_overdue` still asks the real clock, so with a fake date of 1 December 2026 a book due on 20 November isn't reported overdue (it will be wrong whenever the real date differs). Fix the one line so it prints `True False`.

In [ ]:
from datetime import date

class Library:
    def __init__(self, today):
        self.today = today

    def is_overdue(self, due):
        return due < date.today()

lib = Library(today=lambda: date(2026, 12, 1))
print(lib.is_overdue(date(2026, 11, 20)), lib.is_overdue(date(2026, 12, 5)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"using the clock given\"\n    expect_output(\"True False\")\n    from datetime import date as d\n    assert ns[\"Library\"](lambda: d(2020, 1, 1)).is_overdue(d(2026, 11, 20)) is False")

### Bug hunt: An error nobody catches

`load_all` catches `ValueError` from any store, as the `Store` docstring promises. The new `ApiStore` raises `KeyError` instead, so the program crashes. Fix `ApiStore` so it keeps the promise, and the program prints `['rice', None]`.

In [ ]:
class Store:
    def load(self, key):
        """The value for key. Raises ValueError if there's no such key."""
        raise NotImplementedError

class DictStore(Store):
    def __init__(self, data):
        self.data = data

    def load(self, key):
        if key not in self.data:
            raise ValueError(f"no {key}")
        return self.data[key]

class ApiStore(Store):
    def __init__(self, data):
        self.data = data

    def load(self, key):
        return self.data[key]

def load_all(stores, key):
    results = []
    for s in stores:
        try:
            results.append(s.load(key))
        except ValueError:
            results.append(None)
    return results

print(load_all([DictStore({"a": "rice"}), ApiStore({})], "a"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a kept promise\"\n    expect_output(\"['rice', None]\")")

### Puzzle: Passing in a sender

Arrange the lines so that `Alarm` uses whatever sender it's given, and a test passes a fake. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Which principle?

For each description, give the letter of the SOLID principle it breaks: `"S"`, `"O"`, `"L"`, `"I"` or `"D"`.

In [ ]:
# p: A Report class calculates figures, draws charts and emails the result.
# q: A ReadOnlyAccount subclass raises an error in deposit(), which every Account supports.
# r: A Checkout class creates its own real payment gateway inside __init__, so tests charge real cards.
# s: A Bird interface forces every bird class to write fly(), swim() and sing().
# t: Every new discount type means adding another elif to the same price() function.

p = ___
q = ___
r = ___
s = ___
t = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"S\", \"L\", \"D\"), \"p to r\"\n    assert (s, t) == (\"I\", \"O\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a testable fines notifier

A library charges 20p a day for overdue books, up to £5 per book, and sends members a message. Write it so every dependency is passed in:

- `Loan`, a frozen dataclass: `member: str`, `title: str`, `due: date`.
- `LoanSource`, a Protocol with one method, `loans() -> list[Loan]`.
- `fine(loan, today, per_day=20, cap=500)`: a pure function giving the fine in pence (0 if not overdue; a book due yesterday is 1 day late).
- `FineNotifier(source, today, send, per_day=20, cap=500)`: `source` matches `LoanSource`, `today` is a function returning a `date`, and `send` is a function taking `(member, message)`. `run()` works out every member's overdue books, sends **one** message per member who owes anything, like `Ada: 2 overdue, £1.40 to pay (Aké, Kindred)` (titles in alphabetical order), in alphabetical order of members, and returns the total of all fines.
- `MemoryLoans(loans)`: a simple `LoanSource` holding a list, for tests and examples.

`FineNotifier` must not call `date.today()` or print anything itself.

In [ ]:
from dataclasses import dataclass
from datetime import date, timedelta
from typing import Protocol

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def setup():\n    from datetime import date as d\n    L, M = ns[\"Loan\"], ns[\"MemoryLoans\"]\n    return M([L(\"Bo\", \"Small Island\", d(2026, 11, 1)), L(\"Ada\", \"Kindred\", d(2026, 11, 2)), L(\"Ada\", \"Ak\u00e9\", d(2026, 11, 6)),\n              L(\"Chi\", \"Purple Hibiscus\", d(2026, 11, 20)), L(\"Bo\", \"Things Fall Apart\", d(2026, 9, 1))])\ndef test_fine():\n    \"fines\"\n    from datetime import date as d\n    L = ns[\"Loan\"]\n    today = d(2026, 11, 9)\n    assert fine(L(\"x\", \"y\", d(2026, 11, 8)), today) == 20 and fine(L(\"x\", \"y\", d(2026, 11, 9)), today) == 0 and fine(L(\"x\", \"y\", d(2026, 12, 1)), today) == 0\n    assert fine(L(\"x\", \"y\", d(2026, 9, 1)), today) == 500 and fine(L(\"x\", \"y\", d(2026, 11, 2)), today, per_day=10, cap=50) == 50\ndef test_notifier():\n    \"the notifier\"\n    from datetime import date as d\n    F = ns[\"FineNotifier\"]\n    sent = []\n    n = F(setup(), lambda: d(2026, 11, 9), lambda member, msg: sent.append((member, msg)))\n    total = n.run()\n    assert sent == [(\"Ada\", \"Ada: 2 overdue, \u00a32.00 to pay (Ak\u00e9, Kindred)\"), (\"Bo\", \"Bo: 2 overdue, \u00a36.60 to pay (Small Island, Things Fall Apart)\")], f\"sent {sent}\"\n    assert total == 860\n    sent.clear()\n    assert F(setup(), lambda: d(2026, 10, 1), lambda m, s: sent.append(s)).run() == 500 and len(sent) == 1\n    assert \"date.today()\" not in source and \"print(\" not in source")

## Warm-up

`pay_rent` was written for accounts and tested with accounts. A `FixedSaver` **is** an `Account` as far as Python is concerned, so it can be passed in, but it breaks a promise every account makes, and `pay_rent` fails in a way its author never planned for. That's what the first of today's principles is about.

## Liskov substitution

The **Liskov substitution principle** (Barbara Liskov, 1987) says: **if code works with objects of a class, it must keep working when given objects of any subclass.** Week 30, Day 3 met this as "a good override", and mypy's `override` check (Week 33) catches the signature part of it. The full principle is about behaviour as well as signatures. A child must:

- accept at least everything the parent accepts (no extra conditions on the arguments);
- return what the parent promises (the same kind of result, within the same limits);
- raise only the kinds of error the parent's callers already expect;
- keep the parent's rules about its own state (if a parent's balance never goes below zero, neither does the child's).

`FixedSaver` breaks the third point: callers of `withdraw` don't expect a `RuntimeError`. Adding `if isinstance(account, FixedSaver)` to `pay_rent` would "fix" it, but every function that withdraws would need the same check, which is the open-closed problem from yesterday. The real fix is in the design. Here, the parent's contract can say that a withdrawal may be refused, with a specific exception that every caller handles:

Now `FixedSaver` keeps the parent's promise: "a withdrawal may be refused with `WithdrawalRefused`". The other common fix is to stop inheriting: if a fixed saver can't do what an account does, perhaps it isn't one, and a separate class (or a smaller shared parent with only the common methods) describes it honestly.

## When is a square not a rectangle?

A classic example in discussions of this principle shows that "is a" in everyday words isn't enough; the behaviour must fit too. Every square is a rectangle, mathematically. But look at a rectangle class whose sides can be changed:

Code written for rectangles expects `set_width` to leave the height alone; a square can't keep that promise. The usual answers are to make shapes **immutable** (with no `set_width`, a square is a perfectly good rectangle, as in Week 30), or not to inherit at all. The lesson isn't about squares: it's that a subclass relationship is a promise about behaviour, and the promise has to hold.

## Interface segregation

The **interface segregation principle** says: **no code should be forced to depend on methods it doesn't use.** In practice: keep interfaces small, and have each function ask only for what it needs.

Here's the problem with a large interface:

`HomePrinter` must pretend to scan and fax, and anyone with an `OfficeDevice` can't trust that `scan` works, which breaks Liskov substitution too. Small Protocols (Week 32) fix both problems: each describes one ability, devices have whichever abilities they really have, and functions ask for exactly the ability they use:

The same idea applies to plain functions: a function that only reads text should ask for something with a `read` method, not for a whole file object with fifty methods. Smaller requirements mean more things can be passed in, and fewer changes affect the function.

## Dependency inversion

The **dependency inversion principle** says: **high-level code (the rules and decisions that are the point of the program) shouldn't depend on low-level details (the clock, files, databases, email). Both should depend on abstractions.** In Python, that mostly means one habit: **pass dependencies in, instead of creating or naming them inside.**

Here's a reminder service that depends directly on the details:

```python
from datetime import datetime

class Reminders:
    def __init__(self):
        self.items = []

    def run(self):
        now = datetime.now()                     # the real clock, named directly
        for when, text in self.items:
            if when <= now:
                send_email("me@example.org", text)   # a real email, named directly
```

How would you test that a reminder due at 9:00 is sent at 9:01 but not at 8:59? You'd have to wait, or change your computer's clock, and every test run would send real emails. The rules ("send what's due, once") are tangled with the details. Inverting the dependencies means the class receives a clock and a sender, and doesn't care what they really are:

`sent.append` and a lambda returning a fixed time are **fakes**: tiny stand-ins for the real things, which let a test set up any situation instantly. The class now depends only on "something that tells the time" and "something that sends a text", abstractions described by their signatures (`Callable[[], datetime]`, `Callable[[str], None]`), not on `datetime.now` or email. The "inversion" is that the details now fit the shape the high-level code asks for, instead of the high-level code being built around the details.

## Wiring it together

If every class has its dependencies passed in, something has to create the real ones. That happens in one place, at the very edge of the program, usually the `main` function (Week 21) or the code under `if __name__ == "__main__":`:

```python
def main():
    reminders = Reminders(clock=datetime.now, send=email_sender("me@example.org"))
    reminders.add(...)
    reminders.run()
```

This place, sometimes called the **composition root**, is the only code that knows which real clock, file or server is used. Everything else is testable with fakes. A sensible default (`clock=datetime.now`) is fine for convenience, as long as callers can still pass something else.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a class in your own code that calls `datetime.now()`, opens a file, or prints, deep inside its methods, and sketch how it would look with that dependency passed in. Tomorrow: three short principles with big effects, DRY, KISS and YAGNI.